In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Paggamit ng workload

<span id="usage"></span>
Ang usage ay isang sukatan ng tagal ng panahon na naka-lock ang QPU para sa iyong workload, at ito ay kinakalkula nang iba-iba depende sa execution mode na ginagamit mo.

* Ang session usage ay ang wall-clock time habang aktibo ang session. Tingnan ang [Session length](/guides/run-jobs-session#session-length) para sa karagdagang impormasyon tungkol sa transition ng status ng session.
* Ang batch usage ay ang kabuuan ng quantum time (oras na ginugol ng QPU complex sa pagproseso ng iyong job) ng lahat ng job sa batch.
* Ang single job usage ay ang quantum time na ginagamit ng job sa pagproseso.

Tandaan na ang mga nabigo o nakanselong job ay maaaring mabilang sa iyong usage sa ilang partikular na sitwasyon — tingnan ang seksyong [Mga nabigo at nakanselong job](#failed-job) para sa mga detalye.

Para sa mga gumagamit ng bayad na plano, tinutukoy ng usage kung magkano ang halaga ng workload. Tingnan ang [Manage cost](/guides/manage-cost) para sa mga detalye.

<span id="failed-job"></span>
## Usage para sa mga nabigo at nakanselong job
Kapag nabigo o nakansela ang isang job, ang iniuulat na usage ay ang sumusunod:

* Job o batch mode: Ang iniuulat na usage ay ang oras na naka-lock ang QPU para sa pagpapatupad ng iyong workload hanggang sa oras na nabigo o nakansela ito. Kaya naman, kung nangyari ang pagkabigo o pagkakansela bago ang lock, ang iniuulat na usage ay zero. Kung hindi, ang iniuulat na usage ng workload ay ang halaga ng usage bago nabigo o nakansela ang workload. Kaya, ang ilang nabigong job ay hindi lumalabas sa iyong iniuulat na usage at ang iba ay lumalabas.

* Session mode: Ang iniuulat na usage ay ang wall-clock time habang aktibo ang session, anuman ang bilang ng mga job na nabigo o nakansela.

<span id="view-usage"></span>
## I-query ang aktwal na usage ng isang workload
Pagkatapos makumpleto ang isang workload, may ilang paraan upang makita ang aktwal nitong usage:

- Patakbuhin ang [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) o [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) sa `qiskit-ibm-runtime` 0.30 o mas bago. Kung gumagamit ng mas lumang bersyon ng `qiskit-ibm-runtime` (>= 0.23 at < 0.30), mahahanap pa rin ang usage sa `session.details()["usage_time"]` at `batch.details()["usage_time"]`.
- Gamitin ang [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails) para makita ang usage para sa isang partikular na batch o session.
- Gamitin ang [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById) para makita ang usage para sa isang solong job.

<span id="instance-usage"></span>
## Tingnan ang usage ng instance
Maaari mong tingnan ang usage ng isang instance sa pahina ng [Instances](https://quantum.cloud.ibm.com/instances), o, para sa mga may tamang awtoridad, sa pahina ng [Analytics](https://quantum.cloud.ibm.com/analytics). Tandaan na maaaring magpakita ang mga pahinang ito ng iba't ibang bilang ng usage dahil naiiba ang paraan ng pagkalkula ng bawat isa.

Ipinapakita ng pahina ng Instances ang real-time na usage para sa nakaraang 28 araw (rolling), hanggang sa kasalukuyang oras sa kasalukuyang araw. Ang usage sa pahina ng Analytics ay muling kinakalkula bawat oras at kinabibilangan ng huling 28 buong araw; iyon ay, ipinapakita nito ang usage mula 00:00 ng 28 araw na nakalipas hanggang ngayon, sa simula ng oras.

## Tantiyahin ang usage bago mag-submit ng job
Habang ang pagkuha ng tumpak na lokal na pagtatantya ay kumplikado dahil sa mga karagdagang operasyong ginagawa para sa error suppression at mitigation, maaari mong gamitin ang baseline na formula na ito para makakuha ng tinatayang usage:

`<per sub-job overhead> + (rep_delay + <circuit length>) * <num executions>`

- Ang `<per sub-job overhead>` ay isang overhead na halos 2s bawat sub-job. Kasama dito ang mga operasyon tulad ng pag-load ng payload sa control electronics. Maaaring hatiin ang iyong primitive job sa maraming sub-job kung ito ay masyadong malaki para iproseso nang lahat sa isang pagkakataon ng execution engine.
- Ang `rep_delay` ay isang opsyong [maaaring i-customize ng gumagamit](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay), at ang default ay ibinibigay ng `backend.default_rep_delay`, na 250 microseconds sa karamihan ng IBM Quantum backend. Tandaan na ang pagbaba ng `rep_delay` ay nagpapababa ng kabuuang oras ng pagpapatupad ng QPU, ngunit sa gastos ng mas mataas na error rate sa state preparation; tingnan ang gabay na [Dynamic repetition rate execution](/guides/repetition-rate-execution) para sa karagdagang impormasyon.
- Ang `<circuit length>` ay ang kabuuang haba ng instruction. Bawat instruction ay gumagamit ng iba't ibang tagal ng oras sa QPU, kaya nagbabago ang kabuuang haba mula sa isang circuit patungo sa isa pa. Ang isang measurement, halimbawa, ay maaaring 56 beses na mas matagal kaysa sa isang `x` gate. Ang `backend.target[<instruction>][<qubit>].duration` ay maaaring gamitin para mahanap ang eksaktong tagal ng bawat instruction. Ang karaniwang haba ng circuit ay malamang nasa pagitan ng 50-100 microseconds. Kung gumagamit ka ng mga teknik na error suppression o mitigation sa mga primitive, maaaring may mga karagdagang instruction na idinagdag sa iyong circuit, na magpapataas ng kabuuang haba ng circuit.
    > **Note:** Ang [experimental na opsyong `scheduler_timing`](/guides/visualize-circuit-timing) ay nagbabalik ng kabuuang oras ng circuit, ngunit ito ay HINDI ang oras na ginagamit para sa billing.
- Ang `<num executions>` ay ang kabuuang bilang ng mga circuit na pinarami sa bilang ng mga shot, kung saan ang mga circuit ay ang mga nabuong matapos i-broadcast ang mga PUB element.
    - Kung gumagamit ka ng mga teknik na error mitigation sa mga primitive, maaaring may mga karagdagang circuit na tatakbo bilang bahagi ng proseso ng mitigation, na magpapataas ng kabuuang bilang ng mga execution. Bukod pa rito, ang mga advanced na teknik na error mitigation tulad ng PEA at PEC ay may mas mataas na overhead dahil kailangan nilang magpatakbo ng mga circuit para sa noise learning.
    - Ang Estimator ay naggrugrupo ng mga qubit-wise commuting observable, na nagbabawas ng bilang ng mga execution.

Kung hindi ka gumagamit ng anumang advanced na teknik na error mitigation o custom na `rep_delay`, maaari mong gamitin ang `2+0.00035*<num executions>` bilang mabilis na formula.

## Mga susunod na hakbang
> **Tip:** - Suriin ang mga tip na ito: [Minimize job run time](minimize-time).
>     - Itakda ang [Maximum execution time](max-execution-time).
>     - Alamin kung paano mag-transpile nang lokal sa seksyong [Transpile](./transpile/).
>     - Subukan ang gabay na [Compare transpiler settings](/guides/circuit-transpilation-settings).